# NB2 — Ablation Part 1  (CNN-only | α=0.5 | α=0.6)

GPU time: ~60-75 min. Run AFTER NB1. Trains 3 ablation variants. Saves logits + result JSONs to Drive.

**Before running:** Connect T4 GPU (Runtime → Change runtime type → T4 GPU)

**Drive folder:** `MyDrive/CNN_GNN_Results/` must exist (created automatically on first run).

In [ ]:
import os, hashlib, random, warnings, json, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (confusion_matrix, roc_auc_score,
                             f1_score, accuracy_score)
from scipy.stats import chi2
import warnings; warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

CLASS_NAMES  = ['Cardiomegaly','Covid-19','Normal',
                'Pneumonia','Pneumothorax','Tuberculosis']
NUM_CLASSES  = 6
DATASET_PATH = '/content/drive/MyDrive/Dataset'
RESULTS_DIR  = '/content/drive/MyDrive/CNN_GNN_Results'
CKPT_PATH    = f'{RESULTS_DIR}/cnn_gnn_final.pth'
BATCH_SIZE   = 32
IMAGE_SIZE   = 224
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Setup complete.')


Mounted at /content/drive
Device: cuda
Setup complete.


In [ ]:
# ── EXACT model from Untitled7.ipynb ─────────────────────────
class FeatureBasedGNN(nn.Module):
    def __init__(self, num_classes=6, feature_dim=256, hidden_dim=128):
        super().__init__()
        self.disease_prototypes = nn.Parameter(
            torch.randn(num_classes, hidden_dim) * 0.01)
        self.relation_matrix = nn.Parameter(
            torch.randn(num_classes, num_classes) * 0.01)
        self.image_projector = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(), nn.Dropout(0.2))
        self._adj = None

    def forward(self, feat):
        img_feat   = self.image_projector(feat)
        sim        = torch.matmul(img_feat, self.disease_prototypes.t())
        proto_sim  = torch.matmul(self.disease_prototypes,
                                  self.disease_prototypes.t())
        adj        = torch.sigmoid(self.relation_matrix + 0.1 * proto_sim)
        adj_norm   = adj / (adj.sum(dim=1, keepdim=True) + 1e-8)
        self._adj  = adj_norm.detach().cpu()
        propagated = torch.matmul(sim, adj_norm)
        return 0.7 * sim + 0.3 * propagated   # raw logits

    def get_relationship_matrix(self):
        if self._adj is not None: return self._adj.numpy()
        with torch.no_grad():
            proto_sim = torch.matmul(self.disease_prototypes,
                                     self.disease_prototypes.t())
            adj = torch.sigmoid(self.relation_matrix + 0.1 * proto_sim)
            adj_norm = adj / (adj.sum(dim=1, keepdim=True) + 1e-8)
        return adj_norm.cpu().numpy()


class CompleteMedicalModel(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        mobilenet    = models.mobilenet_v2(pretrained=True)
        self.encoder = mobilenet.features
        self.gap     = nn.AdaptiveAvgPool2d((1, 1))
        for i, block in enumerate(self.encoder):
            for p in block.parameters():
                p.requires_grad = (i >= 15)
        self.feature_proj = nn.Sequential(
            nn.Linear(1280, 512), nn.BatchNorm1d(512),
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.BatchNorm1d(256),
            nn.ReLU(), nn.Dropout(0.2))
        self.gnn = FeatureBasedGNN(num_classes, 256, 128)

    def forward(self, x):
        feat = self.encoder(x)
        feat = self.gap(feat).flatten(1)
        feat = self.feature_proj(feat)
        return self.gnn(feat)   # raw logits

print('Proposed model class ready.')


Proposed model class ready.


In [ ]:
# ── Ablation model variants ───────────────────────────────────

# A0: CNN-only — same MobileNetV2 encoder, plain head, no GNN
class CNNOnlyModel(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        mobilenet    = models.mobilenet_v2(pretrained=True)
        self.encoder = mobilenet.features
        self.gap     = nn.AdaptiveAvgPool2d((1, 1))
        for i, block in enumerate(self.encoder):
            for p in block.parameters():
                p.requires_grad = (i >= 15)
        self.head = nn.Sequential(
            nn.Linear(1280,512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512,256),  nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, num_classes))
    def forward(self, x):
        return self.head(self.gap(self.encoder(x)).flatten(1))


# A1/A2/A4: GNN with different alpha (direct vs propagated weighting)
class GNNAlpha(nn.Module):
    def __init__(self, num_classes=6, alpha=0.5):
        super().__init__()
        self.alpha = alpha
        mobilenet    = models.mobilenet_v2(pretrained=True)
        self.encoder = mobilenet.features
        self.gap     = nn.AdaptiveAvgPool2d((1, 1))
        for i, block in enumerate(self.encoder):
            for p in block.parameters():
                p.requires_grad = (i >= 15)
        self.feature_proj = nn.Sequential(
            nn.Linear(1280,512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512,256),  nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.2))
        self.disease_prototypes = nn.Parameter(
            torch.randn(num_classes, 128) * 0.01)
        self.relation_matrix = nn.Parameter(
            torch.randn(num_classes, num_classes) * 0.01)
        self.projector = nn.Sequential(
            nn.Linear(256,128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2))

    def forward(self, x):
        feat       = self.gap(self.encoder(x)).flatten(1)
        feat       = self.feature_proj(feat)
        img_feat   = self.projector(feat)
        sim        = torch.matmul(img_feat, self.disease_prototypes.t())
        proto_sim  = torch.matmul(self.disease_prototypes,
                                  self.disease_prototypes.t())
        adj        = torch.sigmoid(self.relation_matrix + 0.1 * proto_sim)
        adj_norm   = adj / (adj.sum(dim=1, keepdim=True) + 1e-8)
        propagated = torch.matmul(sim, adj_norm)
        return self.alpha * sim + (1 - self.alpha) * propagated


# A5: GNN without learnable relation matrix — only prototype similarity
class GNNNoRelationMatrix(GNNAlpha):
    def __init__(self, num_classes=6):
        super().__init__(num_classes, alpha=0.7)
        self.relation_matrix.data.zero_()
        self.relation_matrix.requires_grad = False

print('Ablation model classes ready.')


Ablation model classes ready.


In [ ]:
# ── Dataset (same transforms as Untitled7) ───────────────────
class ChestXrayDataset(Dataset):
    def __init__(self, root_dir, transform, split):
        self.transform   = transform
        self.image_paths = []
        self.labels      = []
        self.filename_hashes = []   # for filename-collision sanity check (NOT a patient-ID check)
        split_path = os.path.join(root_dir, split)
        print(f'Loading [{split}]...')
        for idx, cls in enumerate(CLASS_NAMES):
            d = os.path.join(split_path, cls)
            if not os.path.exists(d): continue
            files = [f for f in os.listdir(d)
                     if f.lower().endswith(('.png','.jpg','.jpeg'))]
            for f in files:
                self.image_paths.append(os.path.join(d, f))
                self.labels.append(idx)
                self.filename_hashes.append(hashlib.md5(f.encode()).hexdigest()[:8])
            print(f'  {cls:<16}: {len(files)}')
        print(f'  Total: {len(self.image_paths)}')

    def __len__(self): return len(self.image_paths)

    def __getitem__(self, i):
        try: return self.transform(
            Image.open(self.image_paths[i]).convert('RGB')), self.labels[i]
        except: return torch.zeros(3, IMAGE_SIZE, IMAGE_SIZE), 0


train_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
val_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

train_ds = ChestXrayDataset(DATASET_PATH, train_tf, 'train')
val_ds   = ChestXrayDataset(DATASET_PATH, val_tf,   'valid')
test_ds  = ChestXrayDataset(DATASET_PATH, val_tf,   'test')
train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Filename-collision sanity check across splits.
# IMPORTANT: this hashes the FILENAME STRING only, not file content and not a
# real patient identifier. A nonzero count here typically means class folders
# reuse generic filenames (e.g. "1.png") across splits and is NOT evidence of
# image or patient leakage. The actual leakage safeguard used in the paper is
# the separate full-file MD5 content hash reported in the Limitations section.
tr_p = set(train_ds.filename_hashes)
va_p = set(val_ds.filename_hashes)
te_p = set(test_ds.filename_hashes)
print(f'Filename-collision check (informational only, NOT a leakage check) — '
      f'Train∩Val:{len(tr_p&va_p)}  Train∩Test:{len(tr_p&te_p)}  Val∩Test:{len(va_p&te_p)}')
print('Nonzero counts are expected if filenames repeat across class folders '
      'and do not indicate patient or image leakage on their own.')
print('Dataset ready.')


In [ ]:
# ── Loss, metrics, training helpers ─────────────────────────
class MultiClassFocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__(); self.gamma = gamma
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction='none')
        return ((1 - torch.exp(-ce)) ** self.gamma * ce).mean()


def get_metrics(logits_t, tgts_t):
    probs = F.softmax(logits_t, dim=1)
    preds = probs.argmax(1).numpy(); t = tgts_t.numpy()
    acc  = accuracy_score(t, preds)
    mf1  = f1_score(t, preds, average='macro', zero_division=0)
    pf1  = f1_score(t, preds, average=None,
                    labels=list(range(NUM_CLASSES)), zero_division=0)
    cm   = confusion_matrix(t, preds, labels=list(range(NUM_CLASSES)))
    sens, spec = [], []
    for c in range(NUM_CLASSES):
        TP=cm[c,c]; FN=cm[c,:].sum()-TP; FP=cm[:,c].sum()-TP; TN=cm.sum()-TP-FN-FP
        sens.append(TP/(TP+FN+1e-8)); spec.append(TN/(TN+FP+1e-8))
    oh = F.one_hot(tgts_t, NUM_CLASSES).numpy(); auc = []
    for c in range(NUM_CLASSES):
        try: auc.append(roc_auc_score(oh[:,c], probs[:,c].numpy()))
        except: auc.append(float('nan'))
    return {'acc':acc,'mf1':mf1,'pf1':pf1,
            'sens':np.array(sens),'spec':np.array(spec),
            'auc':auc,'cm':cm,'probs':probs}


@torch.no_grad()
def evaluate(model, loader):
    model.eval(); all_logits, all_tgts = [], []
    for imgs, lbls in loader:
        all_logits.append(model(imgs.to(device)).cpu())
        all_tgts.append(lbls)
    logits = torch.cat(all_logits); tgts = torch.cat(all_tgts)
    return logits, tgts, get_metrics(logits, tgts)


def print_test_results(m, label='TEST'):
    print(f'\n{"="*55}\n{label}\n{"="*55}')
    print(f'Accuracy : {m["acc"]*100:.2f}%')
    print(f'Macro F1 : {m["mf1"]:.4f}')
    print(f'Mean AUC : {np.nanmean(m["auc"]):.4f}')
    print(f'Mean Sens: {m["sens"].mean():.4f}')
    print(f'Mean Spec: {m["spec"].mean():.4f}')
    print(f'\n{"Disease":<16} {"F1":>7} {"AUC":>7} {"Sens":>7} {"Spec":>7}')
    print('-'*46)
    for i, name in enumerate(CLASS_NAMES):
        print(f'{name:<16} {m["pf1"][i]:>7.4f} {m["auc"][i]:>7.4f} '
              f'{m["sens"][i]:>7.4f} {m["spec"][i]:>7.4f}')


class EarlyStopping:
    def __init__(self, patience=10):
        self.patience=patience; self.counter=0; self.best=float('inf')
    def __call__(self, v):
        if v < self.best-1e-4: self.best=v; self.counter=0; return False
        self.counter += 1; return self.counter >= self.patience


def quick_train(model, name, epochs=30, patience=10, lr=0.0001):
    """
    Train any model variant and save best checkpoint + result JSON.
    Matches Untitled7 training setup exactly.
    """
    save_path = f'/content/{name}.pth'
    criterion = MultiClassFocalLoss(gamma=2.0)
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    stopper   = EarlyStopping(patience)
    best_f1   = 0.0

    for ep in range(1, epochs+1):
        # ── Train ──
        model.train()
        for imgs, lbls in tqdm(train_loader, desc=f'  Train E{ep:02d}', leave=False):
            loss = criterion(model(imgs.to(device)), lbls.to(device))
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()

        # ── Validate ──
        val_logits, val_tgts, vm = evaluate(model, val_loader)
        val_loss = MultiClassFocalLoss()(val_logits, val_tgts).item()

        print(f'  E{ep:02d}  val_f1={vm["mf1"]:.4f}  '
              f'val_acc={vm["acc"]*100:.2f}%  val_loss={val_loss:.4f}')

        if vm['mf1'] > best_f1:
            best_f1 = vm['mf1']
            torch.save(model.state_dict(), save_path)
            print(f'       -> Best saved')

        if stopper(val_loss):
            print(f'  Early stop at epoch {ep}'); break

    # Load best weights
    model.load_state_dict(
        torch.load(save_path, map_location=device, weights_only=True))

    # Test evaluation
    logits, tgts, m = evaluate(model, test_loader)
    print_test_results(m, label=f'{name} TEST')

    # Save JSON result summary
    result = {
        'model': name,
        'acc':   float(m['acc']),
        'mf1':   float(m['mf1']),
        'auc':   float(np.nanmean(m['auc'])),
        'sens':  float(m['sens'].mean()),
        'spec':  float(m['spec'].mean()),
        'trainable_params': sum(p.numel() for p in model.parameters()
                                if p.requires_grad)
    }
    with open(f'{RESULTS_DIR}/{name}_result.json', 'w') as f:
        json.dump(result, f, indent=2)

    # Copy checkpoint to Drive
    shutil.copy(save_path, f'{RESULTS_DIR}/{name}.pth')
    print(f'  Saved {name}.pth + {name}_result.json to Drive.')
    return model, logits, tgts, m


print('All helpers ready.')


All helpers ready.


In [ ]:
# ── A0: CNN-only (no GNN) ─────────────────────────────────────────────────
print('\n' + '='*55)
print('A0: CNN-only (no GNN)')
print('='*55)
abl_cnn_only = CNNOnlyModel(NUM_CLASSES).to(device)
tr_abl_cnn_only = sum(p.numel() for p in abl_cnn_only.parameters() if p.requires_grad)
print(f'Trainable params: {tr_abl_cnn_only:,}')

abl_cnn_only, lg_abl_cnn_only, tg_abl_cnn_only, m_abl_cnn_only = quick_train(
    abl_cnn_only, 'abl_cnn_only', epochs=30, patience=10)

# Save logits for McNemar test in NB4
torch.save({'logits': lg_abl_cnn_only, 'tgts': tg_abl_cnn_only},
           f'{RESULTS_DIR}/abl_cnn_only_logits.pth')
del abl_cnn_only; torch.cuda.empty_cache()
print(f'A0: CNN-only (no GNN) done.')



A0: CNN-only (no GNN)
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 173MB/s]


Trainable params: 2,316,358


  E01  val_f1=0.9134  val_acc=91.19%  val_loss=0.0984
       -> Best saved


  E02  val_f1=0.9400  val_acc=93.93%  val_loss=0.0617
       -> Best saved


...

  E18  val_f1=0.9795  val_acc=97.93%  val_loss=0.0265


  E19  val_f1=0.9802  val_acc=98.00%  val_loss=0.0251


  E20  val_f1=0.9737  val_acc=97.33%  val_loss=0.0307


  E21  val_f1=0.9737  val_acc=97.33%  val_loss=0.0325


  E22  val_f1=0.9787  val_acc=97.85%  val_loss=0.0260
  Early stop at epoch 22

abl_cnn_only TEST
Accuracy : 98.44%
Macro F1 : 0.9845
Mean AUC : 0.9995
Mean Sens: 0.9844
Mean Spec: 0.9969

Disease               F1     AUC    Sens    Spec
----------------------------------------------
Cardiomegaly      1.0000  1.0000  1.0000  1.0000
Covid-19          0.9727  0.9986  0.9511  0.9991
Normal            0.9612  0.9988  0.9911  0.9858
Pneumonia         0.9865  0.9999  0.9778  0.9991
Pneumothorax      0.9889  0.9999  0.9867  0.9982
Tuberculosis      0.9978  1.0000  1.0000  0.9991
  Saved abl_cnn_only.pth + abl_cnn_only_result.json to Drive.
A0: CNN-only (no GNN) done.


In [ ]:
# ── A1: GNN alpha=0.5 ─────────────────────────────────────────────────
print('\n' + '='*55)
print('A1: GNN alpha=0.5')
print('='*55)
abl_alpha05 = GNNAlpha(NUM_CLASSES, alpha=0.5).to(device)
tr_abl_alpha05 = sum(p.numel() for p in abl_alpha05.parameters() if p.requires_grad)
print(f'Trainable params: {tr_abl_alpha05:,}')

abl_alpha05, lg_abl_alpha05, tg_abl_alpha05, m_abl_alpha05 = quick_train(
    abl_alpha05, 'abl_alpha05', epochs=30, patience=10)

# Save logits for McNemar test in NB4
torch.save({'logits': lg_abl_alpha05, 'tgts': tg_abl_alpha05},
           f'{RESULTS_DIR}/abl_alpha05_logits.pth')
del abl_alpha05; torch.cuda.empty_cache()
print(f'A1: GNN alpha=0.5 done.')



A1: GNN alpha=0.5
Trainable params: 2,348,772


  E01  val_f1=0.9100  val_acc=90.89%  val_loss=0.3505
       -> Best saved


  E02  val_f1=0.9451  val_acc=94.44%  val_loss=0.1271
       -> Best saved


  E03  val_f1=0.9498  val_acc=94.96%  val_loss=0.0880
       -> Best saved


  E04  val_f1=0.9721  val_acc=97.19%  val_loss=0.0440
       -> Best saved


...

  E24  val_f1=0.9765  val_acc=97.63%  val_loss=0.0271


  E25  val_f1=0.9758  val_acc=97.56%  val_loss=0.0286


  E26  val_f1=0.9758  val_acc=97.56%  val_loss=0.0345


  E27  val_f1=0.9758  val_acc=97.56%  val_loss=0.0300


  E28  val_f1=0.9771  val_acc=97.70%  val_loss=0.0264
  Early stop at epoch 28

abl_alpha05 TEST
Accuracy : 98.30%
Macro F1 : 0.9830
Mean AUC : 0.9993
Mean Sens: 0.9830
Mean Spec: 0.9966

Disease               F1     AUC    Sens    Spec
----------------------------------------------
Cardiomegaly      1.0000  1.0000  1.0000  1.0000
Covid-19          0.9589  0.9975  0.9333  0.9973
Normal            0.9590  0.9988  0.9867  0.9858
Pneumonia         0.9933  1.0000  0.9956  0.9982
Pneumothorax      0.9866  0.9998  0.9822  0.9982
Tuberculosis      1.0000  1.0000  1.0000  1.0000
  Saved abl_alpha05.pth + abl_alpha05_result.json to Drive.
A1: GNN alpha=0.5 done.


In [ ]:
# ── A2: GNN alpha=0.6 ─────────────────────────────────────────────────
print('\n' + '='*55)
print('A2: GNN alpha=0.6')
print('='*55)
abl_alpha06 = GNNAlpha(NUM_CLASSES, alpha=0.6).to(device)
tr_abl_alpha06 = sum(p.numel() for p in abl_alpha06.parameters() if p.requires_grad)
print(f'Trainable params: {tr_abl_alpha06:,}')

abl_alpha06, lg_abl_alpha06, tg_abl_alpha06, m_abl_alpha06 = quick_train(
    abl_alpha06, 'abl_alpha06', epochs=30, patience=10)

# Save logits for McNemar test in NB4
torch.save({'logits': lg_abl_alpha06, 'tgts': tg_abl_alpha06},
           f'{RESULTS_DIR}/abl_alpha06_logits.pth')
del abl_alpha06; torch.cuda.empty_cache()
print(f'A2: GNN alpha=0.6 done.')



A2: GNN alpha=0.6
Trainable params: 2,348,772


  E01  val_f1=0.9394  val_acc=93.93%  val_loss=0.2756
       -> Best saved


  E02  val_f1=0.9366  val_acc=93.85%  val_loss=0.1206


  E03  val_f1=0.9381  val_acc=93.93%  val_loss=0.0910


  E04  val_f1=0.9626  val_acc=96.22%  val_loss=0.0560
       -> Best saved


...

  E26  val_f1=0.9802  val_acc=98.00%  val_loss=0.0195


  E27  val_f1=0.9802  val_acc=98.00%  val_loss=0.0201


  E28  val_f1=0.9794  val_acc=97.93%  val_loss=0.0206


  E29  val_f1=0.9780  val_acc=97.78%  val_loss=0.0225


  E30  val_f1=0.9802  val_acc=98.00%  val_loss=0.0215

abl_alpha06 TEST
Accuracy : 98.59%
Macro F1 : 0.9860
Mean AUC : 0.9996
Mean Sens: 0.9859
Mean Spec: 0.9972

Disease               F1     AUC    Sens    Spec
----------------------------------------------
Cardiomegaly      1.0000  1.0000  1.0000  1.0000
Covid-19          0.9752  0.9986  0.9600  0.9982
Normal            0.9612  0.9990  0.9911  0.9858
Pneumonia         0.9911  1.0000  0.9867  0.9991
Pneumothorax      0.9888  0.9999  0.9778  1.0000
Tuberculosis      1.0000  1.0000  1.0000  1.0000
  Saved abl_alpha06.pth + abl_alpha06_result.json to Drive.
A2: GNN alpha=0.6 done.


In [ ]:
# ── Partial results table ─────────────────────────────────────
rows = []
for name, fname in [('A0: CNN-only','abl_cnn_only'),
                    ('A1: GNN α=0.5','abl_alpha05'),
                    ('A2: GNN α=0.6','abl_alpha06')]:
    p = f'{RESULTS_DIR}/{fname}_result.json'
    if os.path.exists(p):
        with open(p) as f: d = json.load(f)
        rows.append({'Model':name,'Acc(%)':f'{d["acc"]*100:.2f}',
                     'Macro F1':f'{d["mf1"]:.4f}','Mean AUC':f'{d["auc"]:.4f}',
                     'Sens':f'{d["sens"]:.4f}','Spec':f'{d["spec"]:.4f}',
                     'Params(M)':f'{d["trainable_params"]/1e6:.2f}'})
df = pd.DataFrame(rows)
print('\nABLATION PART 1 RESULTS:')
print(df.to_string(index=False))
df.to_csv(f'{RESULTS_DIR}/ablation_part1.csv', index=False)
print('Saved ablation_part1.csv to Drive.')



ABLATION PART 1 RESULTS:
        Model Acc(%) Macro F1 Mean AUC   Sens   Spec Params(M)
 A0: CNN-only  98.44   0.9845   0.9995 0.9844 0.9969      2.32
A1: GNN α=0.5  98.30   0.9830   0.9993 0.9830 0.9966      2.35
A2: GNN α=0.6  98.59   0.9860   0.9996 0.9859 0.9972      2.35
Saved ablation_part1.csv to Drive.
